# Malaysian LRRK2 analysis - Concordance check of variant genotype by sequencing technology
- **Project:** LRRK2 mutation spectrum and association study in a multi-ethnic cohort of Malaysian Parkinson’s Disease patients
- **Version:** Python/3.10.12
- **Created:** 05-NOVEMBER-2025
- **Last Update:** 12-DECEMBER-2025

## Description
1. Concordance check

## Generate input .raw file

In [11]:
%%bash
WORK_DIR=/home/jupyter/lrrk2_mutation_r11
cd $WORK_DIR

# Do it for NBA, WGS and CENTOGENE file
/home/jupyter/plink1.9 \
--bfile GP2_all_wgs_LRRK2_updated_qced_rm_exon \
--recode A \
--out GP2_all_wgs_LRRK2_updated_qced_rm_exon

PLINK v1.90b6.9 64-bit (4 Mar 2019)            www.cog-genomics.org/plink/1.9/
(C) 2005-2019 Shaun Purcell, Christopher Chang   GNU General Public License v3
Logging to GP2_all_wgs_LRRK2_updated_qced_rm_exon.log.
Options in effect:
  --bfile GP2_all_wgs_LRRK2_updated_qced_rm_exon
  --out GP2_all_wgs_LRRK2_updated_qced_rm_exon
  --recode A

30088 MB RAM detected; reserving 15044 MB for main workspace.
95 variants loaded from .bim file.
2540 people (1411 males, 1129 females) loaded from .fam.
2540 phenotype values loaded from .fam.
Using 1 thread (no multithreaded calculations invoked).
Before main variant filters, 2540 founders and 0 nonfounders present.
Calculating allele frequencies... 10111213141516171819202122232425262728293031323334353637383940414243444546474849505152535455565758596061626364656667686970717273747576777879808182838485868788899091929394959697989 done.
Total genotyping rate is 0.997857.
95 variants and 2540 people pass filters and QC.
Among remaining phenotypes, 1667 a

## Concordance check

In [ ]:
nba = pd.read_csv(f"{WORK_DIR}/GP2_merge_umkl_qced_LRRK2_keep_exonic_rm_dup_renamed_flipped_2.raw", delim_whitespace = True)
wgs = pd.read_csv(f"{WORK_DIR}/GP2_all_wgs_LRRK2_updated_qced_rm_exon.raw", delim_whitespace = True)
cento = pd.read_csv(f"{WORK_DIR}/centogene_exonic_clean_fin.raw", delim_whitespace = True)

In [112]:
def three_platform_summary(nba, wgs, cento, var):

    def get_series(df, var):
        if var in df.columns:
            return pd.to_numeric(df[var], errors="coerce")
        else:
            # variant not present in this platform
            return pd.Series(np.nan, index=df.index)

    g_nba   = get_series(nba, var)
    g_wgs   = get_series(wgs, var)
    g_cento = get_series(cento, var)

    df = (
        nba[["IID"]].assign(NBA=g_nba.values)
        .merge(wgs[["IID"]].assign(WGS=g_wgs.values), on="IID", how="outer")
        .merge(cento[["IID"]].assign(CENTOGENE=g_cento.values), on="IID", how="outer")
    )

    # carrier flags
    nba_carrier   = df["NBA"] > 0
    wgs_carrier   = df["WGS"] > 0
    cento_carrier = df["CENTOGENE"] > 0

    total_carrier = (nba_carrier | wgs_carrier | cento_carrier).sum()
    nba_ct   = nba_carrier.sum()
    wgs_ct   = wgs_carrier.sum()
    cento_ct = cento_carrier.sum()

    # ---------- pairwise concordance ----------
    def pairwise_fmt(g1, g2):
        mask = (~g1.isna()) & (~g2.isna()) & (g1 > 0) & (g2 > 0)
        n_compared = mask.sum()
        if n_compared == 0:
            return np.nan
        n_match = (g1[mask] == g2[mask]).sum()
        conc = (n_match / n_compared) * 100
        return f"{conc:.2f} ({n_match}/{n_compared})"

    conc_nba_wgs   = pairwise_fmt(df["NBA"], df["WGS"])
    conc_nba_cento = pairwise_fmt(df["NBA"], df["CENTOGENE"])
    conc_wgs_cento = pairwise_fmt(df["WGS"], df["CENTOGENE"])

    # ---------- three-way concordance ----------
    mask3 = (
        (df["NBA"] > 0) & (df["WGS"] > 0) & (df["CENTOGENE"] > 0)
        & (~df["NBA"].isna()) & (~df["WGS"].isna()) & (~df["CENTOGENE"].isna())
    )

    n_compared3 = mask3.sum()
    if n_compared3 == 0:
        conc_3 = np.nan
    else:
        n_match3 = (
            (df.loc[mask3, "NBA"] == df.loc[mask3, "WGS"]) &
            (df.loc[mask3, "NBA"] == df.loc[mask3, "CENTOGENE"])
        ).sum()
        conc3 = (n_match3 / n_compared3) * 100
        conc_3 = f"{conc3:.2f} ({n_match3}/{n_compared3})"

    return {
        "Variant": var,
        "Total carrier": total_carrier,
        "Carrier in NBA": nba_ct,
        "Carrier in WGS": wgs_ct,
        "Carrier in CENTOGENE": cento_ct,
        "NBA + WGS concordance check (%)": conc_nba_wgs,
        "NBA + CENTOGENE concordance check (%)": conc_nba_cento,
        "WGS + CENTOGENE concordance check (%)": conc_wgs_cento,
        "NBA + WGS + CENTOGENE concordance check (%)": conc_3
    }



In [113]:
rows = []

all_vars = sorted(set(nba.columns[6:]) | set(wgs.columns[6:]) | set(cento.columns[6:]))

# remove non-variant columns
all_vars = [v for v in all_vars if v != "_merge"]


for var in all_vars:
    rows.append(three_platform_summary(nba, wgs, cento, var))

final_table = pd.DataFrame(rows)

final_table


,Variant,Total carrier,Carrier in NBA,Carrier in WGS,Carrier in CENTOGENE,NBA + WGS concordance check (%),NBA + CENTOGENE concordance check (%),WGS + CENTOGENE concordance check (%),NBA + WGS + CENTOGENE concordance check (%)
0,chr12:40225142:G:C_C,3,0,3,1,NaN,NaN,100.00 (1/1),NaN
1,chr12:40225148:G:A_A,1,0,1,0,NaN,NaN,NaN,NaN
2,chr12:40225561:A:G_G,10,0,10,0,NaN,NaN,NaN,NaN
3,chr12:40232383:A:G_G,1,1,1,0,100.00 (1/1),NaN,NaN,NaN
4,chr12:40235700:T:G_G,2,2,2,0,100.00 (2/2),NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...
102,chr12:40364877:A:C_C,1,0,0,1,NaN,NaN,NaN,NaN
103,chr12:40364996:C:T_T,1,0,1,0,NaN,NaN,NaN,NaN
104,chr12:40367664:G:A_A,11,9,8,2,100.00 (7/7),100.00 (1/1),NaN,NaN
105,chr12:40367668:G:T_T,1,0,1,0,NaN,NaN,NaN,NaN
